In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [40]:
class MLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.intermidate_size = intermediate_size
        self.fc1 = nn.Linear(hidden_size, intermediate_size)
        self.fc2 = nn.Linear(intermediate_size, hidden_size)

    def forward(self, x, tp=False):
        bs, seq_len, hidden_size = x.shape
        if tp:
            slice = int(self.intermidate_size / 2)
            fc1_slices = self.fc1.weight.split(slice, dim=0)
            x1 = F.linear(x, fc1_slices[0])
            x2 = F.linear(x, fc1_slices[1])
            x = F.relu(torch.concat([x1, x2], dim = -1))
            x1, x2 = x.split(slice, dim=2)
            fc2_slices = self.fc2.weight.split(slice, dim=1)
            return F.linear(x1, fc2_slices[1]) + F.linear(x2, fc2_slices[0])

        else:
            return self.fc2(F.relu(self.fc1(x)))

In [57]:
a = torch.arange(0,20).reshape(2,2,5)
a = a.float()
a

tensor([[[ 0.,  1.,  2.,  3.,  4.],
         [ 5.,  6.,  7.,  8.,  9.]],

        [[10., 11., 12., 13., 14.],
         [15., 16., 17., 18., 19.]]])

In [59]:
weights = torch.ones((10,5))
weights = weights.split(5,dim=0)
weights

(tensor([[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]),
 tensor([[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]))

In [63]:
torch.concat((F.linear(a, weights[0]),F.linear(a, weights[1])),dim=-1)


tensor([[[10., 10., 10., 10., 10., 10., 10., 10., 10., 10.],
         [35., 35., 35., 35., 35., 35., 35., 35., 35., 35.]],

        [[60., 60., 60., 60., 60., 60., 60., 60., 60., 60.],
         [85., 85., 85., 85., 85., 85., 85., 85., 85., 85.]]])

2

1
2
3
4


1
[2, 3, 5, 9, 6, 9]
2
[3, 6, 5, 9, 6, 9]
3
[5, 6, 9, 9, 6, 9]
5
[6, 9, 9, 9, 6, 9]
6
[9, 9, 9, 9, 6, 9]
9
[9, 9, 9, 9, 6, 9]
